In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = "/content/drive/MyDrive/spectnt_data/harmonixset"
DRIVE_OUT  = "/content/drive/MyDrive/spectnt_data/outputs"
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
import os, sys, json, random, time, csv, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
sys.path.insert(0, os.path.dirname(os.getcwd()))
from notebooks.utils import *

REPO = "https://github.com/fharookshaik/BTU_RM_So26.git"
if not os.path.isdir("BTU_RM_So26"):
    !git clone {REPO}
%cd BTU_RM_So26
!pip install -q . 2>&1 | tail -5

os.makedirs("data", exist_ok=True)
if not os.path.isdir("data/harmonixset"):
    !git clone https://github.com/urinieto/harmonixset.git data/harmonixset

for sub in ["audio", "features"]:
    src = os.path.join(DRIVE_DATA, sub)
    dst = os.path.join("data/harmonixset", sub)
    os.makedirs(src, exist_ok=True)
    if not os.path.islink(dst):
        !ln -s "{src}" "{dst}"
if not os.path.islink("outputs"):
    !ln -s "{DRIVE_OUT}" outputs

print(f"Annotations: {len(os.listdir('data/harmonixset/dataset/segments'))} segment files")

In [ ]:
AUDIO_DIR = "data/harmonixset/audio"
URL_CSV = "data/harmonixset/dataset/youtube_urls.csv"
os.makedirs(AUDIO_DIR, exist_ok=True)

existing = {os.path.splitext(f)[0] for f in glob.glob(os.path.join(AUDIO_DIR, "*.wav"))}
print(f"Already downloaded: {len(existing)}")

entries = []
with open(URL_CSV) as f:
    for row in csv.DictReader(f):
        if row["File"] not in existing:
            entries.append(row)

if entries:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    def dl(row):
        out = os.path.join(AUDIO_DIR, f'{row["File"]}.%(ext)s')
        subprocess.run(["yt-dlp", "-x", "--audio-format", "wav", "--audio-quality", "0",
            "--postprocessor-args", "ffmpeg:-ac 1 -ar 16000", "-o", out,
            "--no-playlist", "--quiet", "--no-warnings", row["URL"]],
            check=False, capture_output=True, timeout=120)
    with ThreadPoolExecutor(4) as pool:
        for f in as_completed([pool.submit(dl, e) for e in entries]):
            f.result()
    print(f"Done. Total wavs: {len(glob.glob(os.path.join(AUDIO_DIR, '*.wav')))}")
else:
    print("All audio files present")

In [ ]:
songs = get_songs_with_audio()
print(f"Songs with audio: {len(songs)}")
precompute_features(songs, force=False)
print(f"Features ready: {sum(1 for s in songs if os.path.exists(f'data/harmonixset/features/{s}.npy'))}/{len(songs)}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

TARGET_HOP_TIME = 6 * 512 / 16000
CHUNK_FRAMES = int(24.0 / TARGET_HOP_TIME)
CHUNK_HOP_FRAMES = int(3.0 / TARGET_HOP_TIME)
BATCH_SIZE = 128; MAX_EPOCHS = 100; BATCHES_PER_EPOCH = 500
MAX_GRAD_NORM = 1.0; LR = 0.0005; WEIGHT_DECAY = 0.9

songs = get_songs_with_audio()
precompute_features(songs, force=False)
annotations = load_all_annotations()
print(f"Songs: {len(songs)}, Annotations: {len(annotations)}")

print("Loading features into RAM...")
features = {sid: load_features(sid) for sid in songs}
print("Done")

class HarmonicCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d((2, 1))),
            nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d((2, 1))),
            nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d((2, 1))),
            nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d((2, 1))),
            nn.Sequential(nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1))),
            nn.Sequential(nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((1, 1))),
            nn.Sequential(nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d((1, 1))),
        )
        self.fc = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 8))

    def forward(self, x):
        x = x.unsqueeze(1)
        for conv in self.convs:
            x = conv(x)
        return self.fc(x.mean(dim=[2, 3]))

os.makedirs("outputs/models", exist_ok=True)
os.makedirs("outputs/results", exist_ok=True)
n = len(songs); fold_size = n // 4
song_ids = np.array(songs)
all_results = {}

for fold in range(4):
    val_set = set(song_ids[fold * fold_size:(fold + 1) * fold_size])
    train_ids = [s for s in songs if s not in val_set]
    val_ids = [s for s in songs if s in val_set]
    print(f"\n{'='*60}\nFold {fold}: {len(train_ids)} train, {len(val_ids)} val\n{'='*60}")

    chunks = [(sid, s) for sid in train_ids for s in range(0, features[sid].shape[1] - CHUNK_FRAMES + 1, CHUNK_HOP_FRAMES) if features[sid].shape[1] >= CHUNK_FRAMES]

    model = HarmonicCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=2, factor=0.5)
    best_loss = float("inf"); patience = 0

    for epoch in range(MAX_EPOCHS):
        model.train(); random.shuffle(chunks)
        total = min(len(chunks), BATCHES_PER_EPOCH * BATCH_SIZE)
        pbar = tqdm(range(0, total, BATCH_SIZE), desc=f"Epoch {epoch}", leave=False)
        epoch_loss = 0
        for i in pbar:
            batch = chunks[i:i+BATCH_SIZE]
            fts = torch.stack([torch.from_numpy(features[s][:, s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
            b = torch.stack([torch.from_numpy(annotations[s]["boundary"][s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
            fn = torch.stack([torch.from_numpy(annotations[s]["functions"][s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
            c = CHUNK_FRAMES // 2
            opt.zero_grad()
            out = model(fts)
            loss = 0.9 * F.binary_cross_entropy_with_logits(out[:, 0], b[:, c]) + 0.1 * F.binary_cross_entropy_with_logits(out[:, 1:], fn[:, c, :])
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM); opt.step()
            epoch_loss += loss.item() * len(batch)

        model.eval()
        v_chunks = [(sid, s) for sid in val_ids for s in range(0, features[sid].shape[1] - CHUNK_FRAMES + 1, CHUNK_HOP_FRAMES) if features[sid].shape[1] >= CHUNK_FRAMES]
        v_loss = 0
        with torch.no_grad():
            for i in range(0, len(v_chunks), BATCH_SIZE):
                batch = v_chunks[i:i+BATCH_SIZE]
                fts = torch.stack([torch.from_numpy(features[s][:, s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
                b = torch.stack([torch.from_numpy(annotations[s]["boundary"][s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
                fn = torch.stack([torch.from_numpy(annotations[s]["functions"][s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
                out = model(fts)
                c = CHUNK_FRAMES // 2
                loss = 0.9 * F.binary_cross_entropy_with_logits(out[:, 0], b[:, c]) + 0.1 * F.binary_cross_entropy_with_logits(out[:, 1:], fn[:, c, :])
                v_loss += loss.item() * len(batch)
        v_loss /= max(len(v_chunks), 1); epoch_loss /= max(total, 1)
        sched.step(v_loss)
        print(f"Epoch {epoch:3d} | train: {epoch_loss:.4f} | val: {v_loss:.4f} | lr: {opt.param_groups[0]['lr']:.2e}")

        if v_loss < best_loss:
            best_loss = v_loss; patience = 0
            torch.save(model.state_dict(), f"outputs/models/HarmonicCNN_fold{fold}.pt")
        else:
            patience += 1
            if patience >= 4: print(f"Early stopping at epoch {epoch}"); break

    model.load_state_dict(torch.load(f"outputs/models/HarmonicCNN_fold{fold}.pt", weights_only=True))
    model.eval()
    fold_metrics = []
    for sid in val_ids:
        feat = features[sid]; T = feat.shape[1]; half = CHUNK_FRAMES // 2
        bc = np.zeros(T); fc = np.zeros((T, 7)); cnt = np.zeros(T)
        all_chunks = []; positions = []
        for center in range(T):
            s = max(0, center - half); e = min(T, s + CHUNK_FRAMES)
            if e - s < CHUNK_FRAMES: s = max(0, e - CHUNK_FRAMES)
            all_chunks.append(feat[:, s:e]); positions.append(center)
        with torch.no_grad():
            for i in range(0, len(all_chunks), BATCH_SIZE):
                bt = torch.from_numpy(np.stack(all_chunks[i:i+BATCH_SIZE])).float().to(device)
                out = model(bt)
                for j, c in enumerate(positions[i:i+BATCH_SIZE]):
                    bc[c] += out[j, 0].item(); fc[c] += out[j, 1:].cpu().numpy(); cnt[c] += 1
        cnt = np.maximum(cnt, 1)
        segs = postprocess_song(bc / cnt, fc / cnt[:, np.newaxis], annotations[sid]["duration"])
        fold_metrics.append(compute_metrics(annotations[sid]["segments"], segs, annotations[sid]["duration"]))

    avg = {k: float(np.mean([m[k] for m in fold_metrics])) for k in fold_metrics[0]}
    all_results[f"fold_{fold}"] = avg
    print(f"\nFold {fold}:")
    for k, v in avg.items(): print(f"  {k}: {v:.3f}")

overall = {k: float(np.mean([all_results[f][k] for f in all_results])) for k in list(all_results.values())[0]}
print(f"\n{'='*60}\nHarmonicCNN Average:")
for k, v in overall.items(): print(f"  {k}: {v:.3f}")
json.dump({"per_fold": all_results, "average": overall}, open("outputs/results/HarmonicCNN.json", "w"), indent=2)
print(f"\nResults saved to outputs/results/HarmonicCNN.json")

In [ ]:
import glob, json, pandas as pd
results = sorted(glob.glob("outputs/results/*.json"))
for path in results:
    name = os.path.basename(path).replace(".json", "")
    data = json.load(open(path))
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    df = pd.DataFrame(data["per_fold"]).T
    df.index = [f"Fold {i}" for i in range(len(df))]
    print(df.to_string())
    print(f"\nAverage: {data['average']}")